# EDA — CIS 2450 Final Project
Exploratory Data Analysis for the academic paper similarity project.

Goal: understand the shape, distributions, and quirks of our data before building the K-Means model.

## 1. Load Data

In [ ]:
import sqlite3
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DB_PATH = Path("../papers.db")

conn = sqlite3.connect(DB_PATH)

df = pl.read_database(
    query="SELECT * FROM openalex_papers",
    connection=conn
)

conn.close()

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

## 2. Basic Info
Get a high-level sense of the data: column types, how many nulls exist.

In [ ]:
# Column types
print(df.dtypes)
print()

# Null counts per column
print("Null counts:")
print(df.null_count())

In [ ]:
# Summary statistics for numeric columns
df.select(["cited_by_count", "author_count"]).describe()

**TODO (fill in after running):** Write what you notice here. For example:
- Are there nulls in topic columns?
- What is the range of citation counts? Does it seem reasonable?

## 3. Citation Count Distribution
Citation counts are almost always heavily skewed — a few papers get cited a lot, most get cited very little. This matters for modeling because K-Means is sensitive to scale.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

citations = df["cited_by_count"].to_list()

axes[0].hist(citations, bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Citation Count Distribution (raw)")
axes[0].set_xlabel("Citations")
axes[0].set_ylabel("Number of Papers")

import numpy as np
log_citations = [np.log1p(c) for c in citations]  # log1p handles zeros
axes[1].hist(log_citations, bins=40, color="coral", edgecolor="white")
axes[1].set_title("Citation Count Distribution (log scale)")
axes[1].set_xlabel("log(Citations + 1)")
axes[1].set_ylabel("Number of Papers")

plt.tight_layout()
plt.show()

**TODO (fill in after running):** Is the raw distribution skewed? Does log-scaling make it more normal? This tells us whether we should log-transform citations before clustering.

## 4. Author Count Distribution

In [ ]:
author_counts = df["author_count"].to_list()

plt.figure(figsize=(8, 4))
plt.hist(author_counts, bins=30, color="mediumseagreen", edgecolor="white")
plt.title("Author Count Distribution")
plt.xlabel("Number of Authors")
plt.ylabel("Number of Papers")
plt.show()

print(f"Max authors on one paper: {max(author_counts)}")
print(f"Median authors: {sorted(author_counts)[len(author_counts)//2]}")

**TODO (fill in after running):** Are there any papers with suspiciously high author counts (e.g. 100+)? Those could be large physics/bio collaborations that are outliers.

## 5. Topic / Subfield Breakdown
Since we plan to one-hot encode topics, we need to know how many unique topics there are and how evenly distributed they are.

In [ ]:
topic_counts = (
    df.group_by("primary_subfield")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(f"Unique subfields: {df['primary_subfield'].n_unique()}")
print()
print(topic_counts)

In [ ]:
# Bar chart of top 15 subfields
top15 = topic_counts.head(15)

plt.figure(figsize=(10, 5))
plt.barh(top15["primary_subfield"].to_list()[::-1], top15["count"].to_list()[::-1], color="steelblue")
plt.title("Top 15 Subfields")
plt.xlabel("Number of Papers")
plt.tight_layout()
plt.show()

**TODO (fill in after running):** How many unique subfields are there? If there are 50+ rare subfields, we may want to collapse rare ones into an "Other" category to avoid too many one-hot columns (curse of dimensionality).

## 6. Citations vs. Author Count
Do papers with more authors tend to get more citations?

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    df["author_count"].to_list(),
    df["cited_by_count"].to_list(),
    alpha=0.4,
    color="purple"
)
plt.title("Author Count vs. Citations")
plt.xlabel("Author Count")
plt.ylabel("Citations")
plt.show()

In [ ]:
# Pearson correlation
corr = df.select([
    pl.corr("author_count", "cited_by_count").alias("author_citation_corr")
])
print(corr)

**TODO (fill in after running):** Is there a meaningful correlation? A very high correlation could mean these two features are redundant for clustering.

## 7. Missing Value Deep Dive
Check which papers are missing topic info — these rows may need to be dropped before modeling.

In [ ]:
missing_topic = df.filter(pl.col("primary_topic").is_null())
print(f"Papers missing primary_topic: {len(missing_topic)}")

missing_subfield = df.filter(pl.col("primary_subfield").is_null())
print(f"Papers missing primary_subfield: {len(missing_subfield)}")

missing_doi = df.filter(pl.col("doi_normalized").is_null())
print(f"Papers missing DOI: {len(missing_doi)}")

**TODO (fill in after running):** Papers missing topic info can't be one-hot encoded, so they'll likely need to be dropped in preprocessing. Is the number small enough to drop, or do we need a strategy to fill them?

## 8. Summary & Takeaways

**TODO (fill in after running all cells above):**

- **Citation count:** e.g. highly skewed — will need log transformation before clustering
- **Author count:** e.g. mostly 1-10 authors, a few outliers with 50+
- **Topics:** e.g. X unique subfields, top 5 account for Y% of papers — may collapse rare ones
- **Nulls:** e.g. Z papers missing topic info — will drop in preprocessing
- **Correlation:** e.g. author count and citations have correlation of X — low enough to keep both as features